
# PPUT / CLL LSEG-only 日次終値近似レプリケーション

このノートブックは、`lseg.data` から取得するデータのみを使い、Cboe の PPUT / CLL を日次終値ベースで近似再現します。

- **PPUT**: SPX ロング + 月次 5% OTM SPX Put ロング
- **CLL**: SPX ロング + 四半期 5% OTM SPX Put ロング + 月次 10% OTM SPX Call ショート
- **再現系列 1**: `lseg_option_close` — 実オプションの日次 close / bid-ask mid / settlement / last を使用
- **再現系列 2**: `lseg_iv_close` — IV から Black forward 価格を生成
- **比較対象**: LSEG から取得した Cboe 公式 PPUT / CLL 日次終値

厳密な SOQ / VWAP / 11:00 ET ストライク決定値は標準では使いません。Cboe の計算に含まれるこれらの要素を、日次終値または前日終値で近似します。

> 実行前に、下の `CFG` にある Cboe 公式指数 RIC、オプションチェーン RIC、オプション・IV フィールド名を、利用中の LSEG 契約・環境に合わせて調整してください。


In [ ]:

# =============================================================================
# 0. Imports and global configuration
# =============================================================================
from __future__ import annotations

import math
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import norm
except Exception:
    norm = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# -----------------------------------------------------------------------------
# ここを最初に編集してください。
# -----------------------------------------------------------------------------
CFG: Dict[str, Any] = {
    "run": {
        "start_date": "2018-01-01",
        "end_date": None,                  # Noneなら実行日近辺まで。明示するなら "YYYY-MM-DD"
        "output_dir": "./pput_cll_lseg_output",
        "open_lseg_session": True,
        "close_lseg_session_at_end": False,
        "save_csv": True,
        "save_parquet": True,
        "plot": True,
        "strict": True,                    # True: 欠損価格は例外。False: NaNで継続。
    },
    "lseg": {
        # 環境により Cboe 指数 RIC が異なる可能性があります。
        # 代表例として .PPUT / .CLL を初期値にしていますが、取得不可なら変更してください。
        "spx_ric": ".SPX",
        "spxtr_ric": ".SPXTR",
        "cboe_pput_ric": ".PPUT",
        "cboe_cll_ric": ".CLL",
        "daily_field": "TR.PriceClose",
        "frq": "D",
    },
    "calendar": {
        "roll_weekday": 4,                  # Monday=0, Friday=4
        "quarterly_months": [3, 6, 9, 12],
    },
    "dividend": {
        "negative_dividend_policy": "keep", # keep / floor_zero
        "small_negative_threshold": -0.01,
    },
    "strike_selection": {
        "spot_source": "previous_close",   # previous_close / same_day_close
        "pput_put_moneyness": 0.95,
        "cll_put_moneyness": 0.95,
        "cll_call_moneyness": 1.10,
    },
    "options": {
        # LSEG の SPX option chain / search universe は契約・ワークスペースに依存します。
        # 例: 0#SPX*.U, 0#SPX*.C などのチェーンが利用できる環境があります。
        # まず chain_universe を設定し、後続の「オプションマスター確認セル」で列を確認してください。
        "chain_universe": ["0#SPX*.U"],
        "expiry_match_window_days": 3,
        "chunk_size": 80,
        "master_fields": {
            "option_ric": "RIC",                 # 取得結果の列名が違う場合は aliases で吸収されます
            "expiry": "TR.OptionExpiryDate",
            "strike": "TR.OptionStrikePrice",
            "option_type": "TR.OptionType",
            "exchange": "TR.ExchangeName",
            "underlying": "TR.UnderlyingRIC",
        },
        "mark_fields": {
            # 実際のフィールド名は環境で確認して変更してください。
            # 取れないフィールドは None にしても構いません。
            "bid": "TR.BidPrice",
            "ask": "TR.AskPrice",
            "settle": "TR.SettlementPrice",
            "last": "TR.PriceClose",
        },
        "field_aliases": {
            "date": ["Date", "Calc Date", "Trade Date", "Historical Date"],
            "option_ric": ["Instrument", "RIC", "Option RIC", "Derivative RIC", "Security", "Issue RIC"],
            "expiry": ["Expiry", "Expiration", "Expiry Date", "Expiration Date", "TR.OptionExpiryDate", "Option Expiration Date"],
            "strike": ["Strike", "Strike Price", "TR.OptionStrikePrice", "Option Strike Price"],
            "option_type": ["Option Type", "Put Call", "P/C", "Call Put", "TR.OptionType"],
            "bid": ["Bid", "Bid Price", "TR.BidPrice", "BID", "CF_BID"],
            "ask": ["Ask", "Ask Price", "TR.AskPrice", "ASK", "CF_ASK"],
            "settle": ["Settlement", "Settlement Price", "TR.SettlementPrice", "SETTLE", "OPSETL"],
            "last": ["Last", "Last Price", "Price Close", "TR.PriceClose", "Close", "CF_LAST"],
        },
    },
    "iv": {
        # selected_option_iv_field: 選択された option RIC に対して IV field を履歴取得する簡便モード。
        # surface_grid: 別途 fetch_iv_surface_from_lseg() を利用するモード。
        "mode": "selected_option_iv_field",  # selected_option_iv_field / surface_grid / disabled
        "chunk_size": 80,
        "selected_option_fields": {
            "iv": "TR.ImpliedVolatility",
            "forward": None,
            "discount_factor": None,
            "rate": None,
            "dividend_yield": None,
        },
        "field_aliases": {
            "date": ["Date", "Calc Date", "Trade Date", "Historical Date"],
            "option_ric": ["Instrument", "RIC", "Option RIC", "Derivative RIC", "Security"],
            "iv": ["Implied Volatility", "Implied Vol", "TR.ImpliedVolatility", "IV", "IMP_VOL"],
            "forward": ["Forward", "Forward Price", "TR.ForwardPrice"],
            "discount_factor": ["Discount Factor", "DF", "TR.DiscountFactor"],
            "rate": ["Rate", "Risk Free Rate", "TR.RiskFreeRate", "Zero Rate"],
            "dividend_yield": ["Dividend Yield", "TR.DividendYield"],
        },
        "default_rate": 0.04,
        "default_dividend_yield": 0.015,
        "day_count": 365.0,
        "vol_is_percent_threshold": 3.0,      # 3を超えるIVは%表示とみなし /100
    },
    "index": {
        "initial_level_policy": "match_cboe", # match_cboe / base_100
        "base_level": 100.0,
    },
}

OUTPUT_DIR = Path(CFG["run"]["output_dir"]).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")



## 1. LSEG data access helpers

`lseg.data` の返却列名は、フィールド種別・契約・Workbench/Workspace 設定で微妙に変わることがあります。以下では、列名を緩めに標準化するヘルパーを用意します。


In [ ]:

# =============================================================================
# 1. LSEG data access helpers
# =============================================================================

def _norm_name(x: Any) -> str:
    return "".join(ch for ch in str(x).lower() if ch.isalnum())


def _flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = ["__".join(str(y) for y in tup if str(y) != "") for tup in out.columns]
    else:
        out.columns = [str(c) for c in out.columns]
    return out


def _find_col(df: pd.DataFrame, aliases: Sequence[str], required: bool = False, label: str = "") -> Optional[str]:
    cols = list(df.columns)
    norm_to_col = {_norm_name(c): c for c in cols}
    for a in aliases:
        key = _norm_name(a)
        if key in norm_to_col:
            return norm_to_col[key]
    # partial fallback
    for a in aliases:
        key = _norm_name(a)
        for c in cols:
            if key and (key in _norm_name(c) or _norm_name(c) in key):
                return c
    if required:
        raise KeyError(f"Required column not found for {label or aliases}. Available columns: {cols}")
    return None


def _coerce_date(s: pd.Series) -> pd.Series:
    return pd.to_datetime(s, errors="coerce").dt.tz_localize(None).dt.normalize()


def _coerce_numeric(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def import_lseg_and_open_session(cfg: Mapping[str, Any]):
    """Import lseg.data and open a default session."""
    try:
        import lseg.data as ld  # type: ignore
    except Exception as e:
        raise RuntimeError(
            "lseg.data を import できません。実行環境に LSEG Data Library for Python をインストールし、"
            "Workspace / Platform session を利用可能にしてください。"
        ) from e

    if cfg["run"].get("open_lseg_session", True):
        try:
            ld.open_session()
            print("LSEG session opened.")
        except Exception as e:
            raise RuntimeError("ld.open_session() に失敗しました。認証・セッション設定を確認してください。") from e
    return ld


def lseg_get_data(ld, universe: Sequence[str], fields: Sequence[str], start: Optional[str] = None, end: Optional[str] = None, frq: str = "D") -> pd.DataFrame:
    """Wrapper for ld.get_data with historical TR parameters.

    Many TR.* fields are retrieved through get_data(..., parameters={SDate, EDate, Frq}).
    This wrapper intentionally keeps the call simple and visible.
    """
    universe = [u for u in universe if u]
    fields = [f for f in fields if f]
    if not universe:
        raise ValueError("universe is empty")
    if not fields:
        raise ValueError("fields is empty")

    params: Dict[str, Any] = {}
    if start is not None:
        params["SDate"] = start
    if end is not None:
        params["EDate"] = end
    if frq is not None:
        params["Frq"] = frq

    try:
        raw = ld.get_data(universe=universe, fields=fields, parameters=params)
    except TypeError:
        # Some versions expect parameters as positional or may not accept keyword names identically.
        raw = ld.get_data(universe, fields, params)
    if isinstance(raw, tuple):
        # Some APIs return (data, errors)
        raw = raw[0]
    if not isinstance(raw, pd.DataFrame):
        raw = pd.DataFrame(raw)
    return _flatten_columns(raw)


def normalize_single_field_panel(raw: pd.DataFrame, universe: Mapping[str, str], value_aliases: Sequence[str]) -> pd.DataFrame:
    """Normalize an LSEG single-field historical response to date-indexed wide panel."""
    df = _flatten_columns(raw)

    # If already date-indexed and columns represent instruments, accept it.
    if isinstance(df.index, pd.DatetimeIndex):
        out = df.copy()
        out.index = out.index.tz_localize(None).normalize()
        out = out.rename(columns={ric: name for name, ric in universe.items()})
        return out.sort_index()

    date_col = _find_col(df, ["Date", "Calc Date", "Trade Date", "Historical Date"], required=True, label="date")
    inst_col = _find_col(df, ["Instrument", "RIC", "Security"], required=False, label="instrument")

    value_col = _find_col(df, value_aliases, required=False, label="value")
    if value_col is None:
        excluded = {date_col}
        if inst_col:
            excluded.add(inst_col)
        candidates = [c for c in df.columns if c not in excluded]
        numeric_candidates = []
        for c in candidates:
            if _coerce_numeric(df[c]).notna().sum() > 0:
                numeric_candidates.append(c)
        if len(numeric_candidates) == 1:
            value_col = numeric_candidates[0]
        else:
            raise KeyError(f"Cannot infer value column. Candidates={candidates}; numeric={numeric_candidates}; columns={list(df.columns)}")

    df = df.copy()
    df["date"] = _coerce_date(df[date_col])
    df["value"] = _coerce_numeric(df[value_col])

    if inst_col is None:
        if len(universe) != 1:
            raise KeyError("Instrument column not found and universe has multiple RICs.")
        name = next(iter(universe.keys()))
        out = df[["date", "value"]].dropna(subset=["date"]).set_index("date").rename(columns={"value": name})
        return out.sort_index()

    df["instrument"] = df[inst_col].astype(str)
    ric_to_name = {ric: name for name, ric in universe.items()}
    df["name"] = df["instrument"].map(ric_to_name).fillna(df["instrument"])
    out = df.pivot_table(index="date", columns="name", values="value", aggfunc="last").sort_index()
    return out


def fetch_single_field_panel(ld, universe: Mapping[str, str], field: str, start: str, end: Optional[str], frq: str = "D") -> pd.DataFrame:
    raw = lseg_get_data(ld, list(universe.values()), [field], start=start, end=end, frq=frq)
    value_aliases = [field, field.split(".")[-1], "Price Close", "Close", "Value"]
    return normalize_single_field_panel(raw, universe=universe, value_aliases=value_aliases)



## 2. Underlying / 公式指数 / 配当ポイント

配当ポイントは、`.SPX` と `.SPXTR` の日次終値から推定します。

\[
Div_t = S_{t-1}\left(\frac{SPXTR_t}{SPXTR_{t-1}} - \frac{S_t}{S_{t-1}}\right)
\]


In [ ]:

# =============================================================================
# 2. Underlying, official Cboe indices, dividend points
# =============================================================================

def get_end_date(cfg: Mapping[str, Any]) -> Optional[str]:
    return cfg["run"].get("end_date")


def fetch_underlying_daily(ld, cfg: Mapping[str, Any]) -> pd.DataFrame:
    universe = {
        "spx": cfg["lseg"]["spx_ric"],
        "spxtr": cfg["lseg"]["spxtr_ric"],
    }
    panel = fetch_single_field_panel(
        ld,
        universe=universe,
        field=cfg["lseg"]["daily_field"],
        start=cfg["run"]["start_date"],
        end=get_end_date(cfg),
        frq=cfg["lseg"].get("frq", "D"),
    )
    out = panel.rename_axis("date").reset_index()
    required = ["date", "spx", "spxtr"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise KeyError(f"Missing underlying columns: {missing}. Got columns={list(out.columns)}")
    out = out[required].dropna(subset=["date", "spx", "spxtr"]).sort_values("date").reset_index(drop=True)
    out["spx_ret"] = out["spx"].pct_change()
    out["spxtr_ret"] = out["spxtr"].pct_change()
    out["div_points"] = out["spx"].shift(1) * ((out["spxtr"] / out["spxtr"].shift(1)) - (out["spx"] / out["spx"].shift(1)))

    policy = cfg["dividend"].get("negative_dividend_policy", "keep")
    if policy == "floor_zero":
        thr = cfg["dividend"].get("small_negative_threshold", -0.01)
        out.loc[(out["div_points"] < 0) & (out["div_points"] >= thr), "div_points"] = 0.0
    return out


def fetch_official_indices(ld, cfg: Mapping[str, Any]) -> pd.DataFrame:
    universe = {
        "PPUT": cfg["lseg"].get("cboe_pput_ric"),
        "CLL": cfg["lseg"].get("cboe_cll_ric"),
    }
    universe = {k: v for k, v in universe.items() if v and not str(v).startswith("<")}
    if not universe:
        raise ValueError("Cboe official RICs are not configured.")
    panel = fetch_single_field_panel(
        ld,
        universe=universe,
        field=cfg["lseg"]["daily_field"],
        start=cfg["run"]["start_date"],
        end=get_end_date(cfg),
        frq=cfg["lseg"].get("frq", "D"),
    )
    long = panel.rename_axis("date").reset_index().melt(id_vars="date", var_name="index_name", value_name="level")
    long = long.dropna(subset=["date", "level"]).sort_values(["index_name", "date"]).reset_index(drop=True)
    long["series_name"] = "cboe_true"
    long["daily_return"] = long.groupby("index_name")["level"].pct_change()
    return long[["date", "index_name", "series_name", "level", "daily_return"]]



## 3. ロールカレンダー

`.SPX` の終値が存在する日を営業日とし、第3金曜日が非営業日の場合は直前の SPX 営業日にロールします。


In [ ]:

# =============================================================================
# 3. Roll calendar
# =============================================================================

def third_friday(year: int, month: int) -> pd.Timestamp:
    first = pd.Timestamp(year=year, month=month, day=1)
    # weekday: Monday=0, Friday=4
    days_to_friday = (4 - first.weekday()) % 7
    first_friday = first + pd.Timedelta(days=days_to_friday)
    return first_friday + pd.Timedelta(days=14)


def month_range(start: pd.Timestamp, end: pd.Timestamp) -> List[Tuple[int, int]]:
    dates = pd.date_range(start=start.replace(day=1), end=end.replace(day=1), freq="MS")
    return [(d.year, d.month) for d in dates]


def build_roll_schedule(trading_dates: Sequence[pd.Timestamp], cfg: Mapping[str, Any]) -> pd.DataFrame:
    dates = pd.Series(pd.to_datetime(list(trading_dates))).dt.tz_localize(None).dt.normalize().sort_values().drop_duplicates()
    date_index = pd.Index(dates)
    start, end = dates.min(), dates.max()

    rows = []
    for y, m in month_range(start, end):
        tf = third_friday(y, m)
        if tf in date_index:
            roll = tf
        else:
            prev = dates[dates <= tf]
            if prev.empty:
                continue
            roll = prev.iloc[-1]
        rows.append({"target_third_friday": tf, "roll_date": roll, "month": m, "year": y, "is_quarterly": m in cfg["calendar"]["quarterly_months"]})
    out = pd.DataFrame(rows).drop_duplicates(subset=["roll_date"]).sort_values("roll_date").reset_index(drop=True)
    return out


def get_previous_trading_date(trading_dates: Sequence[pd.Timestamp], date: pd.Timestamp) -> Optional[pd.Timestamp]:
    dates = pd.Series(pd.to_datetime(list(trading_dates))).dt.normalize().sort_values()
    prev = dates[dates < pd.Timestamp(date).normalize()]
    if prev.empty:
        return None
    return pd.Timestamp(prev.iloc[-1])


def get_strike_spot(underlying: pd.DataFrame, date: pd.Timestamp, cfg: Mapping[str, Any]) -> Tuple[float, str]:
    date = pd.Timestamp(date).normalize()
    spot_source = cfg["strike_selection"].get("spot_source", "previous_close")
    u = underlying.set_index("date")
    if spot_source == "same_day_close":
        return float(u.loc[date, "spx"]), "same_day_close"
    if spot_source == "previous_close":
        prev = get_previous_trading_date(u.index, date)
        if prev is None:
            raise ValueError(f"No previous trading date for {date.date()}")
        return float(u.loc[prev, "spx"]), "previous_close"
    raise ValueError(f"Unsupported strike spot source: {spot_source}")



## 4. オプションマスター・日次価格・IV の取得

ここは LSEG 環境差が最も出る部分です。最初に `fetch_option_master_lseg()` を実行し、返却列が `option_ric`, `expiry`, `strike`, `option_type` に正しく標準化されていることを確認してください。


In [ ]:

# =============================================================================
# 4. Option master, option marks, IV marks from LSEG
# =============================================================================

def _rename_by_aliases(df: pd.DataFrame, aliases: Mapping[str, Sequence[str]], required: Sequence[str] = ()) -> pd.DataFrame:
    out = _flatten_columns(df)
    rename = {}
    for target, alist in aliases.items():
        col = _find_col(out, alist, required=target in required, label=target)
        if col is not None:
            rename[col] = target
    out = out.rename(columns=rename)
    return out


def normalize_option_type(x: Any) -> Optional[str]:
    if pd.isna(x):
        return None
    s = str(x).strip().upper()
    if s in {"P", "PUT"} or "PUT" in s:
        return "P"
    if s in {"C", "CALL"} or "CALL" in s:
        return "C"
    return None


def normalize_option_master(raw: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:
    aliases = dict(cfg["options"].get("field_aliases", {}))
    # Include configured field names as aliases.
    for target, field in cfg["options"].get("master_fields", {}).items():
        if field:
            aliases.setdefault(target, [])
            aliases[target] = list(aliases[target]) + [field]

    df = _rename_by_aliases(raw, aliases, required=["option_ric", "expiry", "strike", "option_type"])
    out = pd.DataFrame()
    out["option_ric"] = df["option_ric"].astype(str)
    out["expiry"] = pd.to_datetime(df["expiry"], errors="coerce").dt.tz_localize(None).dt.normalize()
    out["strike"] = _coerce_numeric(df["strike"])
    out["option_type"] = df["option_type"].map(normalize_option_type)
    if "exchange" in df.columns:
        out["exchange"] = df["exchange"]
    if "underlying" in df.columns:
        out["underlying"] = df["underlying"]

    out = out.dropna(subset=["option_ric", "expiry", "strike", "option_type"])
    out = out.drop_duplicates(subset=["option_ric", "expiry", "strike", "option_type"]).sort_values(["expiry", "option_type", "strike", "option_ric"])
    out["expiry"] = pd.to_datetime(out["expiry"]).dt.normalize()
    out["strike"] = out["strike"].astype(float)
    return out.reset_index(drop=True)


def fetch_option_master_lseg(ld, cfg: Mapping[str, Any]) -> pd.DataFrame:
    fields = [v for v in cfg["options"].get("master_fields", {}).values() if v]
    universe = cfg["options"].get("chain_universe", [])
    if isinstance(universe, str):
        universe = [universe]
    raw = lseg_get_data(ld, universe=universe, fields=fields, start=None, end=None, frq=cfg["lseg"].get("frq", "D"))
    master = normalize_option_master(raw, cfg)
    print(f"Option master rows: {len(master):,}, unique RICs: {master['option_ric'].nunique():,}")
    display(master.head(20))
    return master


def _chunked(xs: Sequence[str], n: int) -> Iterable[List[str]]:
    xs = list(dict.fromkeys([str(x) for x in xs if pd.notna(x)]))
    for i in range(0, len(xs), n):
        yield xs[i:i+n]


def normalize_option_marks(raw: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:
    aliases = dict(cfg["options"].get("field_aliases", {}))
    for target, field in cfg["options"].get("mark_fields", {}).items():
        if field:
            aliases.setdefault(target, [])
            aliases[target] = list(aliases[target]) + [field]
    df = _rename_by_aliases(raw, aliases, required=["date", "option_ric"])
    out = pd.DataFrame()
    out["date"] = _coerce_date(df["date"])
    out["option_ric"] = df["option_ric"].astype(str)
    for col in ["bid", "ask", "settle", "last"]:
        out[col] = _coerce_numeric(df[col]) if col in df.columns else np.nan
    out = out.dropna(subset=["date", "option_ric"]).drop_duplicates(subset=["date", "option_ric"], keep="last")
    return out.sort_values(["option_ric", "date"]).reset_index(drop=True)


def fetch_option_marks_lseg(ld, option_rics: Sequence[str], cfg: Mapping[str, Any]) -> pd.DataFrame:
    fields = [v for v in cfg["options"].get("mark_fields", {}).values() if v]
    if not fields:
        raise ValueError("No option mark fields configured.")
    start = cfg["run"]["start_date"]
    end = get_end_date(cfg)
    frames = []
    for chunk in _chunked(option_rics, cfg["options"].get("chunk_size", 80)):
        raw = lseg_get_data(ld, universe=chunk, fields=fields, start=start, end=end, frq=cfg["lseg"].get("frq", "D"))
        frames.append(normalize_option_marks(raw, cfg))
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["date", "option_ric", "bid", "ask", "settle", "last"])
    print(f"Option marks rows: {len(out):,}, unique RICs: {out['option_ric'].nunique() if len(out) else 0:,}")
    return out


def normalize_iv_marks(raw: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:
    aliases = dict(cfg["iv"].get("field_aliases", {}))
    for target, field in cfg["iv"].get("selected_option_fields", {}).items():
        if field:
            aliases.setdefault(target, [])
            aliases[target] = list(aliases[target]) + [field]
    df = _rename_by_aliases(raw, aliases, required=["date", "option_ric", "iv"])
    out = pd.DataFrame()
    out["date"] = _coerce_date(df["date"])
    out["option_ric"] = df["option_ric"].astype(str)
    for col in ["iv", "forward", "discount_factor", "rate", "dividend_yield"]:
        out[col] = _coerce_numeric(df[col]) if col in df.columns else np.nan
    out = out.dropna(subset=["date", "option_ric", "iv"]).drop_duplicates(subset=["date", "option_ric"], keep="last")
    return out.sort_values(["option_ric", "date"]).reset_index(drop=True)


def fetch_selected_option_iv_lseg(ld, option_rics: Sequence[str], cfg: Mapping[str, Any]) -> pd.DataFrame:
    fields = [v for v in cfg["iv"].get("selected_option_fields", {}).values() if v]
    if not fields:
        raise ValueError("No IV fields configured.")
    start = cfg["run"]["start_date"]
    end = get_end_date(cfg)
    frames = []
    for chunk in _chunked(option_rics, cfg["iv"].get("chunk_size", 80)):
        raw = lseg_get_data(ld, universe=chunk, fields=fields, start=start, end=end, frq=cfg["lseg"].get("frq", "D"))
        frames.append(normalize_iv_marks(raw, cfg))
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["date", "option_ric", "iv", "forward", "discount_factor", "rate", "dividend_yield"])
    print(f"IV marks rows: {len(out):,}, unique RICs: {out['option_ric'].nunique() if len(out) else 0:,}")
    return out


def fetch_iv_surface_from_lseg(ld, cfg: Mapping[str, Any]) -> pd.DataFrame:
    """Placeholder for environments where SPX IV surface is available as a grid.

    Required output columns:
        date, expiry, strike, iv
    Optional:
        forward, discount_factor, rate, dividend_yield

    Because LSEG surface entitlements and request conventions vary, this function is intentionally explicit.
    Edit this function only; the rest of the notebook can remain unchanged.
    """
    raise NotImplementedError(
        "CFG['iv']['mode']='surface_grid' を使う場合は、利用中の LSEG IV surface API/fields に合わせて "
        "fetch_iv_surface_from_lseg() を実装してください。標準では selected_option_iv_field モードを使います。"
    )



## 5. オプション選択ロジック

- PPUT put: 95% 水準を**下回る**最も高い strike
- CLL put: 95% 水準**以下**の最も高い strike
- CLL call: 110% 水準**以上**の最も低い strike


In [ ]:

# =============================================================================
# 5. Option selection and roll planning
# =============================================================================

@dataclass(frozen=True)
class OptionRef:
    option_ric: str
    expiry: pd.Timestamp
    strike: float
    option_type: str  # P / C

    def to_dict(self, prefix: str = "") -> Dict[str, Any]:
        p = f"{prefix}_" if prefix else ""
        return {
            f"{p}ric": self.option_ric,
            f"{p}expiry": pd.Timestamp(self.expiry).normalize(),
            f"{p}strike": float(self.strike),
            f"{p}type": self.option_type,
        }


class NoOptionSelectedError(RuntimeError):
    pass


def _option_from_row(row: pd.Series) -> OptionRef:
    return OptionRef(
        option_ric=str(row["option_ric"]),
        expiry=pd.Timestamp(row["expiry"]).normalize(),
        strike=float(row["strike"]),
        option_type=str(row["option_type"]),
    )


def choose_option(
    option_master: pd.DataFrame,
    target_expiry: pd.Timestamp,
    option_type: str,
    target_strike: float,
    rule: str,
    cfg: Mapping[str, Any],
) -> OptionRef:
    """Select option from normalized option master.

    rule:
      - below: strictly below target_strike, pick highest strike
      - at_or_below: <= target_strike, pick highest strike
      - at_or_above: >= target_strike, pick lowest strike
    """
    target_expiry = pd.Timestamp(target_expiry).normalize()
    option_type = option_type.upper()
    window = int(cfg["options"].get("expiry_match_window_days", 3))

    m = option_master.copy()
    m["expiry_diff_days"] = (pd.to_datetime(m["expiry"]).dt.normalize() - target_expiry).dt.days.abs()
    m = m[(m["option_type"] == option_type) & (m["expiry_diff_days"] <= window)]
    if m.empty:
        raise NoOptionSelectedError(f"No {option_type} options near expiry={target_expiry.date()} within ±{window} days.")

    min_diff = m["expiry_diff_days"].min()
    m = m[m["expiry_diff_days"] == min_diff]

    if rule == "below":
        c = m[m["strike"] < target_strike]
        if c.empty:
            raise NoOptionSelectedError(f"No {option_type} strike below {target_strike:.2f} for expiry {target_expiry.date()}.")
        best_strike = c["strike"].max()
        c = c[c["strike"] == best_strike]
    elif rule == "at_or_below":
        c = m[m["strike"] <= target_strike]
        if c.empty:
            raise NoOptionSelectedError(f"No {option_type} strike at or below {target_strike:.2f} for expiry {target_expiry.date()}.")
        best_strike = c["strike"].max()
        c = c[c["strike"] == best_strike]
    elif rule == "at_or_above":
        c = m[m["strike"] >= target_strike]
        if c.empty:
            raise NoOptionSelectedError(f"No {option_type} strike at or above {target_strike:.2f} for expiry {target_expiry.date()}.")
        best_strike = c["strike"].min()
        c = c[c["strike"] == best_strike]
    else:
        raise ValueError(f"Unknown selection rule: {rule}")

    c = c.sort_values(["expiry", "strike", "option_ric"])
    return _option_from_row(c.iloc[0])


def next_roll_after(roll_schedule: pd.DataFrame, date: pd.Timestamp, quarterly_only: bool = False) -> Optional[pd.Timestamp]:
    rs = roll_schedule.copy()
    if quarterly_only:
        rs = rs[rs["is_quarterly"]]
    rs = rs[rs["roll_date"] > pd.Timestamp(date).normalize()].sort_values("roll_date")
    if rs.empty:
        return None
    return pd.Timestamp(rs.iloc[0]["roll_date"]).normalize()


def build_pput_roll_plan(underlying: pd.DataFrame, roll_schedule: pd.DataFrame, option_master: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:
    rows = []
    roll_dates = list(pd.to_datetime(roll_schedule["roll_date"]).dt.normalize())
    for d in roll_dates:
        expiry = next_roll_after(roll_schedule, d, quarterly_only=False)
        if expiry is None:
            continue
        try:
            spot, spot_source = get_strike_spot(underlying, d, cfg)
            target = cfg["strike_selection"]["pput_put_moneyness"] * spot
            new_put = choose_option(option_master, expiry, "P", target, "below", cfg)
            rows.append({
                "date": d,
                "index_name": "PPUT",
                "roll_type": "monthly",
                "strike_spot": spot,
                "strike_spot_source": spot_source,
                "target_put_strike": target,
                **new_put.to_dict("new_put"),
            })
        except Exception as e:
            if cfg["run"].get("strict", True):
                raise
            rows.append({"date": d, "index_name": "PPUT", "roll_type": "monthly", "error": str(e)})
    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def build_cll_roll_plan(underlying: pd.DataFrame, roll_schedule: pd.DataFrame, option_master: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:
    rows = []
    current_put: Optional[OptionRef] = None

    for _, r in roll_schedule.sort_values("roll_date").iterrows():
        d = pd.Timestamp(r["roll_date"]).normalize()
        call_expiry = next_roll_after(roll_schedule, d, quarterly_only=False)
        if call_expiry is None:
            continue
        try:
            spot, spot_source = get_strike_spot(underlying, d, cfg)
            call_target = cfg["strike_selection"]["cll_call_moneyness"] * spot
            new_call = choose_option(option_master, call_expiry, "C", call_target, "at_or_above", cfg)

            roll_type = "monthly"
            new_put: Optional[OptionRef] = None
            put_target = cfg["strike_selection"]["cll_put_moneyness"] * spot

            if bool(r["is_quarterly"]):
                put_expiry = next_roll_after(roll_schedule, d, quarterly_only=True)
                if put_expiry is None:
                    continue
                new_put = choose_option(option_master, put_expiry, "P", put_target, "at_or_below", cfg)
                current_put = new_put
                roll_type = "quarterly"
            else:
                # Cross-roll only after a standing put exists.
                if current_put is not None and new_call.strike < current_put.strike:
                    new_put = choose_option(option_master, current_put.expiry, "P", put_target, "at_or_below", cfg)
                    current_put = new_put
                    roll_type = "cross_roll"

            row = {
                "date": d,
                "index_name": "CLL",
                "roll_type": roll_type,
                "is_quarterly": bool(r["is_quarterly"]),
                "strike_spot": spot,
                "strike_spot_source": spot_source,
                "target_put_strike": put_target,
                "target_call_strike": call_target,
                **new_call.to_dict("new_call"),
            }
            if new_put is not None:
                row.update(new_put.to_dict("new_put"))
            if current_put is not None:
                row.update(current_put.to_dict("standing_put_after_roll"))
            rows.append(row)
        except Exception as e:
            if cfg["run"].get("strict", True):
                raise
            rows.append({"date": d, "index_name": "CLL", "roll_type": "unknown", "error": str(e)})

    return pd.DataFrame(rows).sort_values("date").reset_index(drop=True)


def required_option_rics(*plans: pd.DataFrame) -> List[str]:
    cols = []
    for plan in plans:
        if plan is None or plan.empty:
            continue
        cols.extend([c for c in plan.columns if c.endswith("_ric")])
    rics = []
    for plan in plans:
        if plan is None or plan.empty:
            continue
        for c in [x for x in plan.columns if x.endswith("_ric")]:
            rics.extend(plan[c].dropna().astype(str).tolist())
    return sorted(dict.fromkeys(rics))



## 6. Pricing backend

`MarketClosePricingBackend` は実オプション価格、`IvSelectedOptionBackend` は選択オプションに紐づく IV field から理論価格を返します。


In [ ]:

# =============================================================================
# 6. Pricing backends
# =============================================================================

def _std_norm_cdf(x: float) -> float:
    if norm is not None:
        return float(norm.cdf(x))
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def black_forward_price(option_type: str, F: float, K: float, T: float, sigma: float, df: float = 1.0) -> float:
    option_type = option_type.upper()
    F = float(F)
    K = float(K)
    T = max(float(T), 0.0)
    sigma = max(float(sigma), 0.0)
    df = float(df) if pd.notna(df) else 1.0

    if T <= 1e-10 or sigma <= 1e-10:
        intrinsic = max(0.0, F - K) if option_type == "C" else max(0.0, K - F)
        return df * intrinsic
    vol_sqrt = sigma * math.sqrt(T)
    d1 = (math.log(F / K) + 0.5 * sigma * sigma * T) / vol_sqrt
    d2 = d1 - vol_sqrt
    if option_type == "C":
        return df * (F * _std_norm_cdf(d1) - K * _std_norm_cdf(d2))
    if option_type == "P":
        return df * (K * _std_norm_cdf(-d2) - F * _std_norm_cdf(-d1))
    raise ValueError(f"Unknown option_type: {option_type}")


def yearfrac(start: pd.Timestamp, end: pd.Timestamp, day_count: float = 365.0) -> float:
    return max((pd.Timestamp(end).normalize() - pd.Timestamp(start).normalize()).days / float(day_count), 0.0)


class MarketClosePricingBackend:
    name = "lseg_option_close"

    def __init__(self, option_marks: pd.DataFrame, cfg: Mapping[str, Any]):
        marks = option_marks.copy()
        marks["date"] = pd.to_datetime(marks["date"]).dt.normalize()
        self.marks = marks.set_index(["date", "option_ric"]).sort_index()
        self.strict = cfg["run"].get("strict", True)

    def price(self, opt: OptionRef, date: pd.Timestamp, spot: Optional[float] = None) -> Tuple[float, str]:
        key = (pd.Timestamp(date).normalize(), opt.option_ric)
        if key not in self.marks.index:
            if self.strict:
                raise KeyError(f"Missing option mark for {opt.option_ric} on {pd.Timestamp(date).date()}")
            return np.nan, "missing"
        row = self.marks.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[-1]
        bid, ask = row.get("bid", np.nan), row.get("ask", np.nan)
        if pd.notna(bid) and pd.notna(ask) and ask >= bid and ask > 0:
            return float((bid + ask) / 2.0), "mid"
        for col, src in [("settle", "settlement"), ("last", "last")]:
            v = row.get(col, np.nan)
            if pd.notna(v) and float(v) >= 0:
                return float(v), src
        if self.strict:
            raise ValueError(f"No usable option price for {opt.option_ric} on {pd.Timestamp(date).date()}: {row.to_dict()}")
        return np.nan, "missing"


class IvSelectedOptionBackend:
    name = "lseg_iv_close"

    def __init__(self, iv_marks: pd.DataFrame, option_master: pd.DataFrame, underlying: pd.DataFrame, cfg: Mapping[str, Any]):
        self.cfg = cfg
        iv = iv_marks.copy()
        iv["date"] = pd.to_datetime(iv["date"]).dt.normalize()
        self.iv = iv.set_index(["date", "option_ric"]).sort_index()
        m = option_master.drop_duplicates("option_ric").copy()
        m["expiry"] = pd.to_datetime(m["expiry"]).dt.normalize()
        self.master = m.set_index("option_ric")
        self.underlying = underlying.set_index("date").sort_index()
        self.strict = cfg["run"].get("strict", True)

    def price(self, opt: OptionRef, date: pd.Timestamp, spot: Optional[float] = None) -> Tuple[float, str]:
        date = pd.Timestamp(date).normalize()
        if spot is None:
            spot = float(self.underlying.loc[date, "spx"])
        if date >= opt.expiry:
            return (max(0.0, spot - opt.strike) if opt.option_type == "C" else max(0.0, opt.strike - spot)), "intrinsic"

        key = (date, opt.option_ric)
        if key not in self.iv.index:
            if self.strict:
                raise KeyError(f"Missing IV for {opt.option_ric} on {date.date()}")
            return np.nan, "missing_iv"
        row = self.iv.loc[key]
        if isinstance(row, pd.DataFrame):
            row = row.iloc[-1]
        sigma = float(row["iv"])
        if sigma > self.cfg["iv"].get("vol_is_percent_threshold", 3.0):
            sigma = sigma / 100.0

        T = yearfrac(date, opt.expiry, self.cfg["iv"].get("day_count", 365.0))
        rate = row.get("rate", np.nan)
        q = row.get("dividend_yield", np.nan)
        rate = float(rate) if pd.notna(rate) else float(self.cfg["iv"].get("default_rate", 0.04))
        q = float(q) if pd.notna(q) else float(self.cfg["iv"].get("default_dividend_yield", 0.015))

        F = row.get("forward", np.nan)
        if pd.isna(F):
            F = float(spot) * math.exp((rate - q) * T)
        else:
            F = float(F)
        df = row.get("discount_factor", np.nan)
        if pd.isna(df):
            df = math.exp(-rate * T)
        else:
            df = float(df)
        return black_forward_price(opt.option_type, F, opt.strike, T, sigma, df), "iv_black"


class IvSurfaceBackend:
    name = "lseg_iv_close"

    def __init__(self, surface: pd.DataFrame, underlying: pd.DataFrame, cfg: Mapping[str, Any]):
        self.cfg = cfg
        s = surface.copy()
        s["date"] = pd.to_datetime(s["date"]).dt.normalize()
        s["expiry"] = pd.to_datetime(s["expiry"]).dt.normalize()
        s["strike"] = pd.to_numeric(s["strike"], errors="coerce")
        s["iv"] = pd.to_numeric(s["iv"], errors="coerce")
        self.surface = s.dropna(subset=["date", "expiry", "strike", "iv"]).sort_values(["date", "expiry", "strike"])
        self.underlying = underlying.set_index("date").sort_index()
        self.strict = cfg["run"].get("strict", True)

    def _interp_iv(self, date: pd.Timestamp, expiry: pd.Timestamp, strike: float) -> Tuple[float, pd.Series, str]:
        date = pd.Timestamp(date).normalize()
        expiry = pd.Timestamp(expiry).normalize()
        s = self.surface[self.surface["date"] == date]
        if s.empty:
            if self.strict:
                raise KeyError(f"No IV surface on {date.date()}")
            return np.nan, pd.Series(dtype=float), "missing_surface"

        # Prefer exact expiry, otherwise nearest expiry. This is deliberately conservative.
        s = s.copy()
        s["expiry_diff_days"] = (s["expiry"] - expiry).dt.days.abs()
        s = s[s["expiry_diff_days"] == s["expiry_diff_days"].min()]
        flag = "iv_surface_exact_expiry" if s["expiry_diff_days"].iloc[0] == 0 else "iv_surface_nearest_expiry"

        s = s.sort_values("strike")
        xs = s["strike"].to_numpy(dtype=float)
        ys = s["iv"].to_numpy(dtype=float)
        if strike <= xs.min():
            iv = ys[0]
            flag += "_strike_floor"
        elif strike >= xs.max():
            iv = ys[-1]
            flag += "_strike_cap"
        else:
            iv = float(np.interp(strike, xs, ys))
            flag += "_strike_interp"
        ref = s.iloc[(np.abs(xs - strike)).argmin()]
        return float(iv), ref, flag

    def price(self, opt: OptionRef, date: pd.Timestamp, spot: Optional[float] = None) -> Tuple[float, str]:
        date = pd.Timestamp(date).normalize()
        if spot is None:
            spot = float(self.underlying.loc[date, "spx"])
        if date >= opt.expiry:
            return (max(0.0, spot - opt.strike) if opt.option_type == "C" else max(0.0, opt.strike - spot)), "intrinsic"
        sigma, ref, flag = self._interp_iv(date, opt.expiry, opt.strike)
        if pd.isna(sigma):
            return np.nan, flag
        if sigma > self.cfg["iv"].get("vol_is_percent_threshold", 3.0):
            sigma = sigma / 100.0
        T = yearfrac(date, opt.expiry, self.cfg["iv"].get("day_count", 365.0))
        rate = ref.get("rate", np.nan)
        q = ref.get("dividend_yield", np.nan)
        rate = float(rate) if pd.notna(rate) else float(self.cfg["iv"].get("default_rate", 0.04))
        q = float(q) if pd.notna(q) else float(self.cfg["iv"].get("default_dividend_yield", 0.015))
        F = ref.get("forward", np.nan)
        F = float(F) if pd.notna(F) else float(spot) * math.exp((rate - q) * T)
        df = ref.get("discount_factor", np.nan)
        df = float(df) if pd.notna(df) else math.exp(-rate * T)
        return black_forward_price(opt.option_type, F, opt.strike, T, sigma, df), flag



## 7. PPUT / CLL 指数計算エンジン

ロール日は SOQ と VWAP を使わず、以下の終値近似を採用します。

- `SOQ ≈ S_t`
- `SPX VWAP / SVWAP ≈ S_t`
- `option VWAP ≈ option close`

この近似により、ロール当日の新オプションのプレミアムは当日 P&L に直接入れず、翌営業日以降の分母・分子に反映されます。


In [ ]:

# =============================================================================
# 7. Index calculation engines
# =============================================================================

def opt_from_plan_row(row: pd.Series, prefix: str) -> Optional[OptionRef]:
    ric_col, expiry_col, strike_col, type_col = f"{prefix}_ric", f"{prefix}_expiry", f"{prefix}_strike", f"{prefix}_type"
    if ric_col not in row or pd.isna(row.get(ric_col)):
        return None
    return OptionRef(
        option_ric=str(row[ric_col]),
        expiry=pd.Timestamp(row[expiry_col]).normalize(),
        strike=float(row[strike_col]),
        option_type=str(row[type_col]),
    )


def _initial_level(index_name: str, init_date: pd.Timestamp, official: pd.DataFrame, cfg: Mapping[str, Any]) -> float:
    if cfg["index"].get("initial_level_policy", "match_cboe") == "base_100":
        return float(cfg["index"].get("base_level", 100.0))
    x = official[(official["index_name"] == index_name) & (official["date"] == pd.Timestamp(init_date).normalize())]
    if not x.empty and pd.notna(x.iloc[0]["level"]):
        return float(x.iloc[0]["level"])
    return float(cfg["index"].get("base_level", 100.0))


def run_pput_engine(
    underlying: pd.DataFrame,
    pput_plan: pd.DataFrame,
    pricing_backend: Any,
    official: pd.DataFrame,
    cfg: Mapping[str, Any],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    u = underlying.copy().sort_values("date").reset_index(drop=True)
    u["date"] = pd.to_datetime(u["date"]).dt.normalize()
    plan = pput_plan.copy()
    plan["date"] = pd.to_datetime(plan["date"]).dt.normalize()
    plan_by_date = {pd.Timestamp(r["date"]).normalize(): r for _, r in plan.iterrows() if pd.notna(r.get("new_put_ric"))}

    current_put: Optional[OptionRef] = None
    initialized = False
    level = np.nan
    rows = []
    events = []

    for i, row in u.iterrows():
        d = pd.Timestamp(row["date"]).normalize()
        S = float(row["spx"])
        div = 0.0 if pd.isna(row.get("div_points")) else float(row["div_points"])
        is_roll = d in plan_by_date

        if not initialized:
            if is_roll:
                current_put = opt_from_plan_row(plan_by_date[d], "new_put")
                level = _initial_level("PPUT", d, official, cfg)
                initialized = True
                new_price, new_src = pricing_backend.price(current_put, d, S)
                rows.append({
                    "date": d, "index_name": "PPUT", "series_name": pricing_backend.name,
                    "level": level, "daily_return": np.nan, "gross_return": np.nan,
                    "spx_close": S, "div_points": div, "is_roll_date": True, "roll_type": "initial_monthly",
                    "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                    "put_price": new_price, "put_price_source": new_src,
                    "call_ric": np.nan, "call_strike": np.nan, "call_expiry": pd.NaT, "call_price": np.nan,
                })
                ev = plan_by_date[d].to_dict()
                ev.update({"series_name": pricing_backend.name, "event_note": "initial_position"})
                events.append(ev)
            continue

        prev = u.iloc[i-1]
        d_prev = pd.Timestamp(prev["date"]).normalize()
        S_prev = float(prev["spx"])
        assert current_put is not None

        if is_roll:
            old_put = current_put
            old_put_prev, old_src_prev = pricing_backend.price(old_put, d_prev, S_prev)
            old_settle = max(0.0, old_put.strike - S)  # SOQ close proxy
            gross = (S + div + old_settle) / (S_prev + old_put_prev)
            level = level * gross

            new_put = opt_from_plan_row(plan_by_date[d], "new_put")
            current_put = new_put
            new_price, new_src = pricing_backend.price(current_put, d, S)
            row_out = {
                "date": d, "index_name": "PPUT", "series_name": pricing_backend.name,
                "level": level, "daily_return": gross - 1.0, "gross_return": gross,
                "spx_close": S, "div_points": div, "is_roll_date": True, "roll_type": "monthly",
                "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                "put_price": new_price, "put_price_source": new_src,
                "old_put_ric": old_put.option_ric, "old_put_strike": old_put.strike,
                "old_put_prev_price": old_put_prev, "old_put_prev_price_source": old_src_prev,
                "old_put_settle_proxy": old_settle,
                "call_ric": np.nan, "call_strike": np.nan, "call_expiry": pd.NaT, "call_price": np.nan,
            }
            rows.append(row_out)
            ev = plan_by_date[d].to_dict()
            ev.update({
                "series_name": pricing_backend.name,
                "old_put_ric": old_put.option_ric,
                "old_put_strike": old_put.strike,
                "old_put_settle_proxy": old_settle,
                "new_put_price": new_price,
                "new_put_price_source": new_src,
                "gross_return": gross,
            })
            events.append(ev)
        else:
            put_t, put_src_t = pricing_backend.price(current_put, d, S)
            put_prev, put_src_prev = pricing_backend.price(current_put, d_prev, S_prev)
            gross = (S + div + put_t) / (S_prev + put_prev)
            level = level * gross
            rows.append({
                "date": d, "index_name": "PPUT", "series_name": pricing_backend.name,
                "level": level, "daily_return": gross - 1.0, "gross_return": gross,
                "spx_close": S, "div_points": div, "is_roll_date": False, "roll_type": "none",
                "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                "put_price": put_t, "put_price_source": put_src_t,
                "call_ric": np.nan, "call_strike": np.nan, "call_expiry": pd.NaT, "call_price": np.nan,
            })

    return pd.DataFrame(rows), pd.DataFrame(events)


def run_cll_engine(
    underlying: pd.DataFrame,
    cll_plan: pd.DataFrame,
    pricing_backend: Any,
    official: pd.DataFrame,
    cfg: Mapping[str, Any],
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    u = underlying.copy().sort_values("date").reset_index(drop=True)
    u["date"] = pd.to_datetime(u["date"]).dt.normalize()
    plan = cll_plan.copy()
    plan["date"] = pd.to_datetime(plan["date"]).dt.normalize()
    plan_by_date = {pd.Timestamp(r["date"]).normalize(): r for _, r in plan.iterrows() if pd.notna(r.get("new_call_ric"))}

    current_put: Optional[OptionRef] = None
    current_call: Optional[OptionRef] = None
    initialized = False
    level = np.nan
    rows = []
    events = []

    for i, row in u.iterrows():
        d = pd.Timestamp(row["date"]).normalize()
        S = float(row["spx"])
        div = 0.0 if pd.isna(row.get("div_points")) else float(row["div_points"])
        is_roll = d in plan_by_date

        if not initialized:
            if is_roll and plan_by_date[d].get("roll_type") == "quarterly" and pd.notna(plan_by_date[d].get("new_put_ric")):
                current_put = opt_from_plan_row(plan_by_date[d], "new_put")
                current_call = opt_from_plan_row(plan_by_date[d], "new_call")
                level = _initial_level("CLL", d, official, cfg)
                initialized = True
                put_price, put_src = pricing_backend.price(current_put, d, S)
                call_price, call_src = pricing_backend.price(current_call, d, S)
                rows.append({
                    "date": d, "index_name": "CLL", "series_name": pricing_backend.name,
                    "level": level, "daily_return": np.nan, "gross_return": np.nan,
                    "spx_close": S, "div_points": div, "is_roll_date": True, "roll_type": "initial_quarterly",
                    "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                    "put_price": put_price, "put_price_source": put_src,
                    "call_ric": current_call.option_ric, "call_strike": current_call.strike, "call_expiry": current_call.expiry,
                    "call_price": call_price, "call_price_source": call_src,
                })
                ev = plan_by_date[d].to_dict()
                ev.update({"series_name": pricing_backend.name, "event_note": "initial_position"})
                events.append(ev)
            continue

        prev = u.iloc[i-1]
        d_prev = pd.Timestamp(prev["date"]).normalize()
        S_prev = float(prev["spx"])
        assert current_put is not None and current_call is not None

        if is_roll:
            plan_row = plan_by_date[d]
            roll_type = str(plan_row.get("roll_type"))
            old_put, old_call = current_put, current_call
            put_prev, put_src_prev = pricing_backend.price(old_put, d_prev, S_prev)
            call_prev, call_src_prev = pricing_backend.price(old_call, d_prev, S_prev)
            call_settle = max(0.0, S - old_call.strike)  # SOQ close proxy

            if roll_type == "quarterly":
                put_settle = max(0.0, old_put.strike - S)  # SOQ close proxy
                gross = (S + div + put_settle - call_settle) / (S_prev + put_prev - call_prev)
                current_put = opt_from_plan_row(plan_row, "new_put")
                current_call = opt_from_plan_row(plan_row, "new_call")
                current_put_price, current_put_src = pricing_backend.price(current_put, d, S)
                current_call_price, current_call_src = pricing_backend.price(current_call, d, S)
            elif roll_type in {"monthly", "cross_roll"}:
                # old put is marked at close; old call is settled. If cross-roll, put is replaced after close proxy.
                put_t, put_src_t = pricing_backend.price(old_put, d, S)
                gross = (S + div + put_t - call_settle) / (S_prev + put_prev - call_prev)
                if roll_type == "cross_roll" and pd.notna(plan_row.get("new_put_ric")):
                    current_put = opt_from_plan_row(plan_row, "new_put")
                else:
                    current_put = old_put
                current_call = opt_from_plan_row(plan_row, "new_call")
                current_put_price, current_put_src = pricing_backend.price(current_put, d, S)
                current_call_price, current_call_src = pricing_backend.price(current_call, d, S)
                put_settle = np.nan
            else:
                raise ValueError(f"Unknown CLL roll_type: {roll_type}")

            level = level * gross
            rows.append({
                "date": d, "index_name": "CLL", "series_name": pricing_backend.name,
                "level": level, "daily_return": gross - 1.0, "gross_return": gross,
                "spx_close": S, "div_points": div, "is_roll_date": True, "roll_type": roll_type,
                "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                "put_price": current_put_price, "put_price_source": current_put_src,
                "call_ric": current_call.option_ric, "call_strike": current_call.strike, "call_expiry": current_call.expiry,
                "call_price": current_call_price, "call_price_source": current_call_src,
                "old_put_ric": old_put.option_ric, "old_put_strike": old_put.strike,
                "old_call_ric": old_call.option_ric, "old_call_strike": old_call.strike,
                "old_put_prev_price": put_prev, "old_put_prev_price_source": put_src_prev,
                "old_call_prev_price": call_prev, "old_call_prev_price_source": call_src_prev,
                "old_put_settle_proxy": put_settle,
                "old_call_settle_proxy": call_settle,
            })
            ev = plan_row.to_dict()
            ev.update({
                "series_name": pricing_backend.name,
                "old_put_ric": old_put.option_ric,
                "old_put_strike": old_put.strike,
                "old_call_ric": old_call.option_ric,
                "old_call_strike": old_call.strike,
                "old_put_settle_proxy": put_settle,
                "old_call_settle_proxy": call_settle,
                "new_put_price": current_put_price,
                "new_put_price_source": current_put_src,
                "new_call_price": current_call_price,
                "new_call_price_source": current_call_src,
                "gross_return": gross,
            })
            events.append(ev)
        else:
            put_t, put_src_t = pricing_backend.price(current_put, d, S)
            call_t, call_src_t = pricing_backend.price(current_call, d, S)
            put_prev, put_src_prev = pricing_backend.price(current_put, d_prev, S_prev)
            call_prev, call_src_prev = pricing_backend.price(current_call, d_prev, S_prev)
            gross = (S + div + put_t - call_t) / (S_prev + put_prev - call_prev)
            level = level * gross
            rows.append({
                "date": d, "index_name": "CLL", "series_name": pricing_backend.name,
                "level": level, "daily_return": gross - 1.0, "gross_return": gross,
                "spx_close": S, "div_points": div, "is_roll_date": False, "roll_type": "none",
                "put_ric": current_put.option_ric, "put_strike": current_put.strike, "put_expiry": current_put.expiry,
                "put_price": put_t, "put_price_source": put_src_t,
                "call_ric": current_call.option_ric, "call_strike": current_call.strike, "call_expiry": current_call.expiry,
                "call_price": call_t, "call_price_source": call_src_t,
            })

    return pd.DataFrame(rows), pd.DataFrame(events)



## 8. 比較・集計・可視化


In [ ]:

# =============================================================================
# 8. Comparison, metrics, output, plots
# =============================================================================

def add_normalized_level(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values(["index_name", "series_name", "date"])
    def _norm(g: pd.DataFrame) -> pd.DataFrame:
        first = g["level"].dropna().iloc[0] if g["level"].notna().any() else np.nan
        g["normalized_level_100"] = g["level"] / first * 100.0
        return g
    return out.groupby(["index_name", "series_name"], group_keys=False).apply(_norm)


def compute_metrics(all_levels: pd.DataFrame) -> pd.DataFrame:
    df = all_levels.copy()
    rows = []
    for index_name, gi in df.groupby("index_name"):
        true = gi[gi["series_name"] == "cboe_true"][["date", "daily_return", "level"]].rename(columns={"daily_return": "true_return", "level": "true_level"})
        for series_name, gr in gi.groupby("series_name"):
            if series_name == "cboe_true":
                continue
            rep = gr[["date", "daily_return", "level", "is_roll_date"]].rename(columns={"daily_return": "rep_return", "level": "rep_level"})
            x = true.merge(rep, on="date", how="inner").dropna(subset=["true_return", "rep_return"])
            if x.empty:
                continue
            diff = x["rep_return"] - x["true_return"]
            roll = x[x["is_roll_date"] == True]
            nonroll = x[x["is_roll_date"] != True]
            monthly = x.set_index("date")[["true_level", "rep_level"]].resample("ME").last().pct_change().dropna()
            rows.append({
                "index_name": index_name,
                "series_name": series_name,
                "n_obs": len(x),
                "start": x["date"].min(),
                "end": x["date"].max(),
                "daily_return_corr": x["true_return"].corr(x["rep_return"]),
                "monthly_return_corr": monthly["true_level"].corr(monthly["rep_level"]) if len(monthly) > 2 else np.nan,
                "mae_bps": diff.abs().mean() * 1e4,
                "rmse_bps": math.sqrt((diff ** 2).mean()) * 1e4,
                "ann_tracking_error_pct": diff.std(ddof=1) * math.sqrt(252) * 100,
                "mean_error_bps": diff.mean() * 1e4,
                "max_abs_error_bps": diff.abs().max() * 1e4,
                "roll_day_mae_bps": (roll["rep_return"] - roll["true_return"]).abs().mean() * 1e4 if not roll.empty else np.nan,
                "nonroll_day_mae_bps": (nonroll["rep_return"] - nonroll["true_return"]).abs().mean() * 1e4 if not nonroll.empty else np.nan,
                "cumulative_error_pct": (x["rep_level"].iloc[-1] / x["true_level"].iloc[-1] - 1.0) * 100,
            })
    return pd.DataFrame(rows).sort_values(["index_name", "series_name"]).reset_index(drop=True)


def build_active_returns(all_levels: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for index_name, gi in all_levels.groupby("index_name"):
        true = gi[gi["series_name"] == "cboe_true"][["date", "daily_return"]].rename(columns={"daily_return": "cboe_return"})
        for series_name, gr in gi.groupby("series_name"):
            if series_name == "cboe_true":
                continue
            x = true.merge(gr[["date", "daily_return", "is_roll_date", "roll_type"]], on="date", how="inner")
            x["index_name"] = index_name
            x["series_name"] = series_name
            x["active_return"] = x["daily_return"] - x["cboe_return"]
            rows.append(x)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


def plot_normalized_levels(all_levels: pd.DataFrame, index_name: str):
    gi = all_levels[all_levels["index_name"] == index_name].copy()
    if gi.empty:
        return
    pvt = gi.pivot_table(index="date", columns="series_name", values="normalized_level_100", aggfunc="last")
    ax = pvt.plot(figsize=(12, 5), title=f"{index_name}: normalized level = 100")
    ax.set_ylabel("Normalized level")
    ax.grid(True, alpha=0.3)
    plt.show()


def plot_active_returns(active: pd.DataFrame, index_name: str):
    gi = active[active["index_name"] == index_name].copy()
    if gi.empty:
        return
    for series_name, g in gi.groupby("series_name"):
        ax = (g.set_index("date")["active_return"] * 1e4).plot(figsize=(12, 4), title=f"{index_name}: active return bps vs Cboe ({series_name})")
        ax.set_ylabel("bps")
        ax.grid(True, alpha=0.3)
        plt.show()


def save_outputs(all_levels: pd.DataFrame, roll_events: pd.DataFrame, metrics: pd.DataFrame, active: pd.DataFrame, cfg: Mapping[str, Any]):
    outdir = Path(cfg["run"]["output_dir"]).expanduser().resolve()
    outdir.mkdir(parents=True, exist_ok=True)
    if cfg["run"].get("save_csv", True):
        all_levels.to_csv(outdir / "daily_levels.csv", index=False)
        roll_events.to_csv(outdir / "roll_events.csv", index=False)
        metrics.to_csv(outdir / "comparison_metrics.csv", index=False)
        active.to_csv(outdir / "active_returns.csv", index=False)
    if cfg["run"].get("save_parquet", True):
        try:
            all_levels.to_parquet(outdir / "daily_levels.parquet", index=False)
            roll_events.to_parquet(outdir / "roll_events.parquet", index=False)
            metrics.to_parquet(outdir / "comparison_metrics.parquet", index=False)
            active.to_parquet(outdir / "active_returns.parquet", index=False)
        except Exception as e:
            warnings.warn(f"Parquet save skipped: {e}")
    print(f"Saved outputs to: {outdir}")



## 9. Main workflow

このセルを実行すると、LSEG からデータ取得、ロールプラン作成、再現指数計算、比較、保存まで一括で実行します。


In [ ]:

# =============================================================================
# 9. Main workflow
# =============================================================================

def run_workflow(cfg: Mapping[str, Any]) -> Dict[str, pd.DataFrame]:
    ld = import_lseg_and_open_session(cfg)
    try:
        print("Fetching underlying daily data...")
        underlying = fetch_underlying_daily(ld, cfg)
        display(underlying.head())
        display(underlying.tail())

        print("Fetching official Cboe index data...")
        official = fetch_official_indices(ld, cfg)
        display(official.groupby("index_name").tail(3))

        print("Building roll schedule...")
        roll_schedule = build_roll_schedule(underlying["date"], cfg)
        display(roll_schedule.tail(12))

        print("Fetching option master from LSEG...")
        option_master = fetch_option_master_lseg(ld, cfg)

        print("Building PPUT roll plan...")
        pput_plan = build_pput_roll_plan(underlying, roll_schedule, option_master, cfg)
        display(pput_plan.tail(12))

        print("Building CLL roll plan...")
        cll_plan = build_cll_roll_plan(underlying, roll_schedule, option_master, cfg)
        display(cll_plan.tail(12))

        rics = required_option_rics(pput_plan, cll_plan)
        print(f"Required option RICs: {len(rics):,}")
        if len(rics) == 0:
            raise RuntimeError("No option RICs selected. Check option master and roll plans.")

        print("Fetching option EOD marks for selected RICs...")
        option_marks = fetch_option_marks_lseg(ld, rics, cfg)
        display(option_marks.head())

        market_backend = MarketClosePricingBackend(option_marks, cfg)
        print("Running market-close replication...")
        pput_mkt, pput_mkt_events = run_pput_engine(underlying, pput_plan, market_backend, official, cfg)
        cll_mkt, cll_mkt_events = run_cll_engine(underlying, cll_plan, market_backend, official, cfg)

        replicated_frames = [pput_mkt, cll_mkt]
        event_frames = [pput_mkt_events, cll_mkt_events]

        if cfg["iv"].get("mode") != "disabled":
            if cfg["iv"].get("mode") == "selected_option_iv_field":
                print("Fetching selected option IV data from LSEG...")
                iv_marks = fetch_selected_option_iv_lseg(ld, rics, cfg)
                display(iv_marks.head())
                iv_backend = IvSelectedOptionBackend(iv_marks, option_master, underlying, cfg)
            elif cfg["iv"].get("mode") == "surface_grid":
                print("Fetching IV surface grid from LSEG...")
                iv_surface = fetch_iv_surface_from_lseg(ld, cfg)
                display(iv_surface.head())
                iv_backend = IvSurfaceBackend(iv_surface, underlying, cfg)
            else:
                raise ValueError(f"Unknown IV mode: {cfg['iv'].get('mode')}")

            print("Running IV-close replication...")
            pput_iv, pput_iv_events = run_pput_engine(underlying, pput_plan, iv_backend, official, cfg)
            cll_iv, cll_iv_events = run_cll_engine(underlying, cll_plan, iv_backend, official, cfg)
            replicated_frames.extend([pput_iv, cll_iv])
            event_frames.extend([pput_iv_events, cll_iv_events])

        all_levels = pd.concat([official] + replicated_frames, ignore_index=True, sort=False)
        all_levels["date"] = pd.to_datetime(all_levels["date"]).dt.normalize()
        all_levels = add_normalized_level(all_levels)

        roll_events = pd.concat(event_frames, ignore_index=True, sort=False) if event_frames else pd.DataFrame()
        if not roll_events.empty:
            roll_events["date"] = pd.to_datetime(roll_events["date"]).dt.normalize()

        metrics = compute_metrics(all_levels)
        active = build_active_returns(all_levels)

        print("Comparison metrics:")
        display(metrics)
        print("Recent roll events:")
        display(roll_events.tail(20))

        save_outputs(all_levels, roll_events, metrics, active, cfg)

        if cfg["run"].get("plot", True):
            for index_name in ["PPUT", "CLL"]:
                plot_normalized_levels(all_levels, index_name)
                plot_active_returns(active, index_name)

        return {
            "underlying": underlying,
            "official": official,
            "roll_schedule": roll_schedule,
            "option_master": option_master,
            "pput_plan": pput_plan,
            "cll_plan": cll_plan,
            "option_marks": option_marks,
            "all_levels": all_levels,
            "roll_events": roll_events,
            "metrics": metrics,
            "active_returns": active,
        }
    finally:
        if cfg["run"].get("close_lseg_session_at_end", False):
            try:
                ld.close_session()
                print("LSEG session closed.")
            except Exception:
                pass

# 実行
results = run_workflow(CFG)



## 10. 追加分析例

`results` に全中間データを保持しています。以下のセルは任意で実行できます。


In [ ]:

# 年次サマリー例
all_levels = results["all_levels"].copy()
all_levels["year"] = pd.to_datetime(all_levels["date"]).dt.year
annual = (
    all_levels.sort_values("date")
    .groupby(["index_name", "series_name", "year"])
    .agg(start_level=("level", "first"), end_level=("level", "last"), n=("level", "count"))
    .reset_index()
)
annual["annual_return"] = annual["end_level"] / annual["start_level"] - 1.0
display(annual.tail(30))

# ロール日の誤差ランキング例
active = results["active_returns"].copy()
if not active.empty:
    roll_error = active[active["is_roll_date"] == True].copy()
    roll_error["abs_active_bps"] = roll_error["active_return"].abs() * 1e4
    display(roll_error.sort_values("abs_active_bps", ascending=False).head(30))
